# Darcy: primal and mixed convergence

We solve $-\Delta p=2\pi^2p$ with $p=\cos(\pi x)\cos(\pi y)$ and prescribed boundary pressure. This analytical problem appears in the mixed-local Darcy study ([Duran et al., 2019](https://doi.org/10.1016/j.cma.2019.05.013)). Our diagonal macrotriangles, four local subdivisions and degree-zero traces define a separate low-order verification; the paper's tables are not reproduced.

In [ ]:
from pathlib import Path
import sys

# Locate the checkout from the repository root or any notebook subdirectory.
ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "pyproject.toml").is_file()
     and (path / "src" / "pymhm").is_dir()
     and (path / "notebooks").is_dir()),
    None,
)
if ROOT is None:
    raise RuntimeError("Run this notebook from the PyMHM checkout.")
root = ROOT
for directory in (ROOT, ROOT / "examples"):
    if str(directory) not in sys.path:
        sys.path.insert(0, str(directory))


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pymhm import TriangleMesh, SkeletonSpace, FaceSpace
from pymhm._legacy.models.vector import solve_brinkman, solve_elasticity


## Declare the local and global equations

The application writes local A, B and C blocks with the shared FEM kernels,
then supplies them to the generic assembler. Primal coefficients satisfy

$$
\begin{aligned}
a_T(p,v)&=(K\nabla p,\nabla v)_T,\\
L_T(v)&=(f,v)_T+\sum_j\alpha_jv(x_j),\\
B_T\lambda&=\langle\lambda,v\rangle_{\partial T},\\
C_Tp&=-\langle p,\mu\rangle_{\partial T}.
\end{aligned}
$$

Signs follow the fixed macroface normals. RT0 instead uses flux, pressure and
private boundary-pressure coefficients, including the local normal-flux
constraint. The global `Equation` supplies the signed Dirichlet moments and
retains each constant pressure mode. Pure Neumann data add the physical pressure
integral through an explicit constraint. `define_darcy` returns equations;
`assemble`, the global solve and field interpretation are separate stages.


In [ ]:
from pymhm import assemble, SolverConfig
from examples.formulations.darcy import define_darcy, pressure_constraints, recover_darcy


def exact(x):
    """Return the prescribed analytical pressure."""
    return np.cos(np.pi * x[:, 0]) * np.cos(np.pi * x[:, 1])


def source(x):
    """Return the exact negative Laplacian."""
    return 2 * np.pi**2 * exact(x)


errors = {}
for formulation in ("primal", "mixed"):
    errors[formulation] = []
    for n in (2, 4, 8):
        result_definition = define_darcy(
            TriangleMesh.unit_square(n),
            source=source,
            dirichlet=exact,
            formulation=formulation,
            local_refinement=4,
        )
        result_system = assemble(result_definition.problem)
        result = recover_darcy(
            result_definition,
            result_system,
            result_system.solve(constraints=pressure_constraints(result_definition, result_system)),
        )
        errors[formulation].append(result.l2_error(exact))
    assert np.all(np.diff(errors[formulation]) < 0)
    print(
        formulation,
        errors[formulation],
        "rates",
        np.log2(np.array(errors[formulation][:-1]) / errors[formulation][1:]),
    )
    plt.loglog([0.5, 0.25, 0.125], errors[formulation], "o-", label=formulation)
plt.xlabel("Macro grid spacing")
plt.ylabel("Pressure L2 error")
plt.legend()
plt.grid(True)
plt.show()


P0 mixed pressure and P1 primal pressure have different approximation orders. Compare physical flux separately; pressure error alone cannot establish local conservation.